# A model with one named beam
**COMPAS foundations · AEC Formats: Model**

**Why this is needed:** a mesh stores geometry. A model organizes named building elements and their relationships. Here one beam belongs to one floor. This is a small COMPAS Model, not an IFC file or a structural analysis.

**You already know:** Box, Mesh and JSON. **New ideas:** an element and a group. Dimensions are in metres. Run the cells from top to bottom.


In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps


In [ ]:
%pip install -q compas_model==0.9.3


In [ ]:
try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()


## 1. Make one element and put it in a group
A `BeamElement` has width, depth, length and a name. Its length follows its local **Z axis**. `Model` owns the elements; `Ground floor` is their organizational parent.


In [ ]:
from compas_model.models import Model
from compas_model.elements import BeamElement

model = Model()
floor = model.add_group("Ground floor")
beam = model.add_element(
    BeamElement(width=0.2, depth=0.3, length=3.0, name="B1"),
    parent=floor,
)
print([(type(element).__name__, element.name) for element in model.elements()])


## 2. View the beam's geometry
After the element joins the model, `beam.modelgeometry` gives its mesh in model coordinates. The group has no shape, so we draw only the beam.


In [ ]:
from compas_notebook import Viewer
from compas_notebook.config import Config

config = Config()
config.view.width = 700
config.view.height = 360
config.view.camera.position = [4, -6, 3]
config.view.camera.target = [0, 0, 1.5]
viewer = Viewer(config=config)
viewer.scene.add(beam.modelgeometry, color="#5552bb", show_edges=True)
viewer.show()


## 3. Save and restore the model
COMPAS JSON preserves the group, the element type and the beam dimensions. Use **File → Save and Export Notebook As → Notebook** for your notebook, and Jupyter's file browser to download the JSON file created below.


In [ ]:
from pathlib import Path
from compas import json_dumps, json_loads

session = {"units": "m", "model": model}
payload = json_dumps(session, pretty=True)
Path("one-beam.compas.json").write_text(payload)
restored = json_loads(payload)
restored_beams = [element for element in restored["model"].elements()
                  if isinstance(element, BeamElement)]
assert len(restored_beams) == 1
assert restored_beams[0].name == beam.name
assert restored_beams[0].length == beam.length
assert restored["units"] == "m"
print("Round-trip:", restored_beams[0].name, restored_beams[0].length, "m")
print("Saved one-beam.compas.json")


## Try one change
1. Change the name to `B2` and the length to 2 m. Predict which displayed dimension changes.
2. Run the element, viewer and JSON cells again. Check the restored name and length.
3. Explain why a floor group and a beam mesh store different information.

**References:** [COMPAS Model 0.9.3](https://pypi.org/project/compas-model/0.9.3/), [Model and BeamElement source](https://github.com/BlockResearchGroup/compas_model), [UCL 2024 data-exchange examples](https://github.com/BRG-teaching/UCL_2024). Original BRG tutorial, using the released COMPAS Model API.
